# Phase 12: ImageNet-Pretrained ResNet-50 U-Net Reproducibility Experiment

**Objective**: Evaluate whether the zero-shot cross-dataset generalization advantage observed in Vision Transformers (Swin-Tiny: $57.50\%$ FG IoU, Swin-MoE: $57.85\%$ FG IoU) over randomly initialized CNNs (Standard U-Net: $50.79\%$, Attention U-Net: $45.35\%$) persists when the convolutional architecture receives identical ImageNet-1K pretraining.

**Primary Research Question**: Does ImageNet-1K pretraining resolve the domain-shift penalty in CNNs, or does Swin-Tiny possess architectural inductive biases (such as hierarchical window self-attention and dynamic spatial receptive fields) that confer cross-dataset robustness independently of pretraining?

**Hardware Accelerator**: Select GPU T4 x2 or P100.
**Attached Datasets**: `busi-dataset` and `busbra-dataset` (or search BUSI / BUSBRA on Kaggle).

---
### Experimental Controls:
1. **Architecture**: Canonical ResNet-50 encoder (ImageNet-1K V1 pretrained) paired with a 5-stage symmetric U-Net decoder with multi-scale skip connections, totaling **32,521,011 parameters** (~$32.52\text{M}$, strictly matched to Swin-Tiny's $29.29\text{M}$ and Standard U-Net's $31.04\text{M}$).
2. **Training Dataset**: Held-in BUSI dataset (80% train, 10% validation, 10% in-domain test across 10 matched seeds).
3. **Zero-Shot Evaluation**: Unseen external BUS-BRA dataset ($N = 1,063$ images) with zero fine-tuning or threshold tuning.
4. **Exact Optimization Parity**: AdamW ($lr=1e-4, wd=0.05$), CosineAnnealingLR ($T_{\max}=50$), batch size 16, Focal Tversky Loss, early stopping patience 10 across 10 matched seeds: `[42, 123, 314, 777, 999, 1001, 2026, 2027, 2718, 9001]`.

In [ ]:
# 1. SETUP ENVIRONMENT & REPO (FRESH CLONE FROM MAIN)
import os, sys, shutil, glob, torch, json

# Ensure clean root in Kaggle
if os.path.exists('/kaggle/working'):
    os.chdir('/kaggle/working')
    !rm -rf /kaggle/working/vision_transformer_research

print('Fetching latest repository from GitHub (main branch)...')
!git clone --depth 1 -b main https://github.com/toqeer-ahmed/vision_tranformer_moe.git vision_transformer_research

repo_dir = '/kaggle/working/vision_transformer_research' if os.path.exists('/kaggle/working/vision_transformer_research') else os.path.abspath('vision_transformer_research')
os.chdir(repo_dir)

sys.path.insert(0, repo_dir)
if 'datasets' in sys.modules:
    del sys.modules['datasets']

!pip install -r requirements.txt -q
!pip install albumentations -q

# Run ResNet-50 U-Net 9-test assertion suite
!python -m unittest tests/test_resnet50_unet.py


In [ ]:
# 2. HIGH-SPEED, ZERO-COPY DATASET MOUNTING
import os, shutil

print('Setting up dataset paths...')
os.makedirs('data/medical_dataset/images', exist_ok=True)
os.makedirs('data/medical_dataset/masks', exist_ok=True)
os.makedirs('data/busbra_dataset/images', exist_ok=True)
os.makedirs('data/busbra_dataset/masks', exist_ok=True)

def link_or_copy(src, dst):
    if os.path.exists(dst):
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy(src, dst)

if os.path.exists('/kaggle/input'):
    for root_dir, _, files in os.walk('/kaggle/input', followlinks=True):
        for file in files:
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                src = os.path.join(root_dir, file)
                if 'busi' in src.lower() and 'checkpoints' not in src.lower():
                    if '_mask' in file.lower():
                        link_or_copy(src, os.path.join('data/medical_dataset/masks', file))
                    else:
                        link_or_copy(src, os.path.join('data/medical_dataset/images', file))
                elif 'busbra' in src.lower():
                    if 'mask' in src.lower():
                        link_or_copy(src, os.path.join('data/busbra_dataset/masks', file))
                    else:
                        link_or_copy(src, os.path.join('data/busbra_dataset/images', file))

print(f"Total BUSI images mounted: {len(os.listdir('data/medical_dataset/images'))}")
print(f"Total BUSBRA images mounted: {len(os.listdir('data/busbra_dataset/images'))}")

from datasets.medical_dataset import MedicalImageMaskDataset, get_medical_transforms
_, transform = get_medical_transforms(224)
busi_dataset = MedicalImageMaskDataset('data/medical_dataset', transform=transform)
busbra_dataset = MedicalImageMaskDataset('data/busbra_dataset', transform=transform)
print(f"Verified BUSI dataset: {len(busi_dataset)} pairs.")
print(f"Verified BUS-BRA dataset: {len(busbra_dataset)} pairs.")


In [ ]:
# 3. TRAINING LOOP ACROSS MATCHED SEEDS (WITH AUTOMATIC DISK PROTECTION)
import os, glob

# 10 Matched seeds: [42, 123, 314, 777, 999, 1001, 2026, 2027, 2718, 9001]
seeds = [42, 123, 314, 777, 999, 1001, 2026, 2027, 2718, 9001]

for seed in seeds:
    print(f"\n======================================")
    print(f"TRAINING RESNET-50 U-NET - SEED {seed}")
    print(f"======================================\n")
    !python training/train_resnet50_unet.py --config configs/phase12_resnet50_unet/resnet50_unet_seed{seed}.yaml
    
    # Remove intermediate last_checkpoint.pth to preserve disk headroom
    for f in glob.glob(f"outputs/resnet50_unet_seed{seed}_*/checkpoints/last_checkpoint.pth"):
        try: os.remove(f)
        except Exception: pass


In [ ]:
# 4. RIGOROUS ZERO-SHOT EVALUATION ON BUS-BRA (N=1,063) & VOLUME RATIO ANALYSIS
import glob, yaml, json, os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from models.resnet50_unet import ResNet50UNet
from evaluation.metrics import SegmentationMetricAccumulator

busbra_loader = DataLoader(busbra_dataset, batch_size=16, shuffle=False, num_workers=2)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def evaluate_resnet50_unet_checkpoint(ckpt_path, config_path):
    with open(config_path, 'r') as f: config = yaml.safe_load(f)
    model_cfg = config['model']
    model = ResNet50UNet(
        num_classes=model_cfg.get('num_classes', 3),
        pretrained=False,
        freeze_encoder=model_cfg.get('freeze_encoder', False),
        freeze_bn=model_cfg.get('freeze_bn', False),
        use_transpose_conv=model_cfg.get('use_transpose_conv', False)
    )
    ckpt = torch.load(ckpt_path, map_location=device)
    state_dict = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt
    model.load_state_dict(state_dict)
    model.to(device).eval()
    
    acc = SegmentationMetricAccumulator(num_classes=2)
    with torch.no_grad():
        for imgs, masks in busbra_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            with torch.amp.autocast('cuda'):
                logits = model(imgs)
            preds = (logits.argmax(dim=1) > 0).long()
            acc.update(preds, masks)
    return acc.compute()

results = {}
data_rows = []

for seed in seeds:
    name = f"resnet50_unet_seed{seed}"
    dirs = sorted(glob.glob(f"outputs/{name}_*"))
    if not dirs:
        print(f"Skipping {name} (no output directory found)")
        continue
    target_dir = dirs[-1]
    ckpt_path = os.path.join(target_dir, 'checkpoints', 'best_model.pth')
    if not os.path.exists(ckpt_path):
        print(f"Skipping {name} (no best_model.pth in {target_dir})")
        continue
        
    cfg_path = f"configs/phase12_resnet50_unet/{name}.yaml"
    print(f"Evaluating {name} on external BUS-BRA (N=1,063)...")
    m = evaluate_resnet50_unet_checkpoint(ckpt_path, cfg_path)
    results[name] = m
    
    pred_vol = m['class_tp'][1] + m['class_fp'][1]
    gt_vol = m['class_tp'][1] + m['class_fn'][1]
    vol_ratio = pred_vol / gt_vol if gt_vol > 0 else 0.0
    
    data_rows.append({
        "Model": "ResNet-50 U-Net (ImageNet-Pretrained)",
        "Seed": seed,
        "mIoU": float(m['mean_iou']),
        "Foreground_IoU": float(m['class_iou'][1]),
        "Background_IoU": float(m['class_iou'][0]),
        "mDice": float(m['mean_dice']),
        "Foreground_Dice": float(m['class_dice'][1]),
        "Background_Dice": float(m['class_dice'][0]),
        "Foreground_Precision": float(m['class_precision'][1]),
        "Foreground_Recall": float(m['class_recall'][1]),
        "Pixel_Accuracy": float(m['pixel_accuracy']),
        "Predicted_Volume_Pixels": int(pred_vol),
        "GroundTruth_Volume_Pixels": int(gt_vol),
        "Volume_Ratio": float(vol_ratio)
    })

os.makedirs('results/phase12_resnet50_unet', exist_ok=True)
with open('results/phase12_resnet50_unet/resnet50_unet_reproducibility_metrics.json', 'w') as f:
    json.dump(results, f, indent=4)

df = pd.DataFrame(data_rows)
df.to_csv('results/phase12_resnet50_unet/RESNET50_UNET_PER_SEED_RESULTS.csv', index=False)

summary_df = df.drop(columns=['Seed']).groupby('Model').agg(['mean', 'std']).round(4)
summary_df.to_csv('results/phase12_resnet50_unet/RESNET50_UNET_SUMMARY_RESULTS.csv')

print("\n" + "="*60)
print("=== RESNET-50 U-NET ZERO-SHOT EVALUATION SUMMARY (BUS-BRA) ===")
print("="*60)
print(summary_df)

print("\n=== PER-SEED DETAILED METRICS TABLE ===")
print(df.to_string(index=False))

# Package results into zip archive using pure Python zipfile
import zipfile
zip_filename = 'results_phase12_resnet50_unet.zip'
print(f"\nCreating {zip_filename} archive...")
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for folder in ['results/phase12_resnet50_unet', 'configs/phase12_resnet50_unet']:
        if os.path.exists(folder):
            for root, _, files in os.walk(folder):
                for file in files:
                    full_p = os.path.join(root, file)
                    zipf.write(full_p, full_p)
    if os.path.exists('outputs'):
        for root, _, files in os.walk('outputs'):
            for file in files:
                if 'resnet50_unet' in root.lower() and (file == 'best_model.pth' or file.endswith(('.json', '.csv', '.png', '.svg', '.txt')) or 'events.out.tfevents' in file):
                    full_p = os.path.join(root, file)
                    zipf.write(full_p, full_p)

if os.path.exists('/kaggle/working'):
    shutil.copy(zip_filename, os.path.join('/kaggle/working', zip_filename))

print(f"\nDone! Download {zip_filename} ({os.path.getsize(zip_filename)/(1024*1024):.2f} MB) from Kaggle output window.")
